In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

customer_schema = StructType([
    StructField("CustomerId", IntegerType(), True),
    StructField("CustomerName", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("UpdatedAt", StringType(), True)
])
bronze_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(customer_schema)
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/bronze/schema/day11_customers/")
    .load("/Volumes/workspace/bronze/customer_data_files/")
)

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/workspace/bronze/checkpoints/day11_customers/")
    .trigger(availableNow=True)
    .toTable("bronze.day11_customers")
)

In [0]:
%sql
SELECT *
FROM bronze.day11_customers
ORDER BY CustomerId;

CustomerId,CustomerName,City,Age,UpdatedAt,_rescued_data
101,Arun,Bangalore,31,2026-08-21 10:00:00,null
101,Arun,Chennai,30,2026-08-20 09:00:00,null
102,Kumar,Hyderabad,36,2026-08-22 09:00:00,null
102,Kumar,Bangalore,35,2026-08-20 09:05:00,null
103,Priya,Chennai,27,2026-08-20 09:10:00,null
104,Ravi,Coimbatore,32,2026-08-21 10:05:00,null
105,Meena,Madurai,29,2026-08-22 09:05:00,null


In [0]:
from pyspark.sql.functions import col, to_timestamp

bronze = spark.table("bronze.day11_customers")

bronze = bronze.withColumn(
    "UpdatedAt",
    to_timestamp(col("UpdatedAt"))
)
display(bronze)

CustomerId,CustomerName,City,Age,UpdatedAt,_rescued_data
101,Arun,Chennai,30,2026-08-20T09:00:00.000Z,null
102,Kumar,Bangalore,35,2026-08-20T09:05:00.000Z,null
103,Priya,Chennai,27,2026-08-20T09:10:00.000Z,null
101,Arun,Bangalore,31,2026-08-21T10:00:00.000Z,null
104,Ravi,Coimbatore,32,2026-08-21T10:05:00.000Z,null
102,Kumar,Hyderabad,36,2026-08-22T09:00:00.000Z,null
105,Meena,Madurai,29,2026-08-22T09:05:00.000Z,null


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = (
    Window
    .partitionBy("CustomerId")
    .orderBy(col("UpdatedAt").desc())
)

latest_customers = (
    bronze
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)
display(latest_customers)

CustomerId,CustomerName,City,Age,UpdatedAt,_rescued_data
101,Arun,Bangalore,31,2026-08-21T10:00:00.000Z,null
102,Kumar,Hyderabad,36,2026-08-22T09:00:00.000Z,null
103,Priya,Chennai,27,2026-08-20T09:10:00.000Z,null
104,Ravi,Coimbatore,32,2026-08-21T10:05:00.000Z,null
105,Meena,Madurai,29,2026-08-22T09:05:00.000Z,null


In [0]:
%sql
CREATE table if not EXISTS silver.day11_customers;


In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(spark, "silver.day11_customers")

(  silver_table.alias("target").merge(
            latest_customers.alias("source"),
            "target.CustomerId = source.CustomerId"
        ).whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
  )

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
SELECT *
FROM silver.day11_customers
ORDER BY CustomerId;

CustomerId,CustomerName,City,Age,UpdatedAt,_rescued_data
101,Arun,Bangalore,31,2026-08-21T10:00:00.000Z,null
102,Kumar,Hyderabad,36,2026-08-22T09:00:00.000Z,null
103,Priya,Chennai,27,2026-08-20T09:10:00.000Z,null
104,Ravi,Coimbatore,32,2026-08-21T10:05:00.000Z,null
105,Meena,Madurai,29,2026-08-22T09:05:00.000Z,null
